# 03 · Modelo Gold – Olist Brazilian E-Commerce
**Origen:** `olist.silver` · **Destino:** `olist.gold`

Gold no es "Silver más bonito": es **una tabla por pregunta de negocio**, ya agregada, para que un tablero la consulte sin hacer cálculos.

| Tabla | Pregunta de negocio |
|---|---|
| `fact_ventas` | Tabla base: una fila por ítem vendido, con cliente, producto, vendedor, entrega y reseña ya unidos |
| `ventas_mensuales` | 1. ¿Cómo evolucionan pedidos, clientes e ingresos mes a mes, y cuál es el mejor mes? |
| `ranking_categorias` | 2. ¿Qué categorías generan más ingresos y cómo las califican los clientes? |
| `entregas_por_estado` | 3. ¿En qué estados se entrega más lento y con menos puntualidad? |
| `satisfaccion_por_retraso` | 4. ¿Cuánto cae la satisfacción cuando el pedido llega tarde? |
| `v_ventas_publico` | Vista sin datos personales del cliente, para consumo general |

**Reglas de Gold**
- Sin `NULL` en dimensiones: `sin_categoria`, `desconocido`. Un `NULL` desaparece de los filtros de un tablero.
- Los pedidos `canceled` y `unavailable` no cuentan como venta.
- Ingresos = `precio + flete`, en reales (BRL).

## 1. Parámetros y esquema

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "olist")
catalogo = dbutils.widgets.get("catalogo")
silver = f"{catalogo}.silver"
gold = f"{catalogo}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold} COMMENT 'Capa Gold: tablas agregadas listas para consumo de negocio'")


def s(tabla):
    return spark.table(f"{silver}.{tabla}")


print("Origen:", silver, "| Destino:", gold)

## 2. Control de calidad previo
Gold solo se construye si la **última ejecución de Silver cuadró** en todas sus tablas (según `silver.log_calidad`).

In [0]:
ultima = spark.sql(f"""
    SELECT tabla, cuadra
    FROM {silver}.log_calidad
    WHERE fecha_proceso = (SELECT max(fecha_proceso) FROM {silver}.log_calidad)""").collect()

fallidas = [r.tabla for r in ultima if not r.cuadra]
if not ultima or fallidas:
    raise Exception(f"Silver no está lista (sin registro o sin cuadrar: {fallidas}). "
                    "Ejecutar 02_transformacion_silver primero.")
print(f"Silver OK: {len(ultima)} tablas reconciliadas en la última ejecución")

## 3. `fact_ventas`: la tabla base
Grano: **una fila por ítem vendido**. Decisiones:
- **Las reseñas se promedian por pedido antes del join.** Un pedido puede tener más de una reseña: unirlas directo duplicaría ítems e inflaría los ingresos.
- La puntualidad se mide **por día** (`to_date`): la fecha estimada viene sin hora, y comparar con la hora exacta marcaría como tarde una entrega del mismo día.
- `dias_retraso` negativo = llegó antes de lo prometido.

In [0]:
ESTADOS_NO_VENTA = ["canceled", "unavailable"]

pedidos = s("pedidos").filter(~F.col("estado_pedido").isin(ESTADOS_NO_VENTA))

resenas_pedido = (s("resenas").groupBy("pedido_id")
                  .agg(F.round(F.avg("puntaje"), 2).alias("puntaje_resena")))

clientes = s("clientes").select(
    "cliente_id", "cliente_unico_id",
    F.col("estado").alias("cliente_estado"),
    F.col("ciudad").alias("cliente_ciudad"))
productos = s("productos").select("producto_id", F.col("categoria_en").alias("categoria"))
vendedores = s("vendedores").select("vendedor_id", F.col("estado").alias("vendedor_estado"))

fact = (s("items_pedido")
        .join(pedidos, "pedido_id", "inner")
        .join(clientes, "cliente_id", "left")
        .join(productos, "producto_id", "left")
        .join(vendedores, "vendedor_id", "left")
        .join(resenas_pedido, "pedido_id", "left")
        .select(
            "pedido_id", "item_num", "estado_pedido",
            "fecha_compra",
            F.date_format("fecha_compra", "yyyy-MM").alias("anio_mes"),
            "cliente_unico_id",
            F.coalesce("cliente_estado", F.lit("desconocido")).alias("cliente_estado"),
            F.coalesce("cliente_ciudad", F.lit("desconocido")).alias("cliente_ciudad"),
            "producto_id",
            F.coalesce("categoria", F.lit("sin_categoria")).alias("categoria"),
            "vendedor_id",
            F.coalesce("vendedor_estado", F.lit("desconocido")).alias("vendedor_estado"),
            "precio", "flete",
            (F.col("precio") + F.col("flete")).alias("total_item"),
            "fecha_entrega", "fecha_entrega_estimada",
            F.datediff("fecha_entrega", "fecha_compra").alias("dias_entrega"),
            F.datediff("fecha_entrega", "fecha_entrega_estimada").alias("dias_retraso"),
            (F.to_date("fecha_entrega") <= F.to_date("fecha_entrega_estimada")).alias("entregado_a_tiempo"),
            "puntaje_resena",
        ))

tabla_fact = f"{gold}.fact_ventas"
fact.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(tabla_fact)
spark.sql(f"COMMENT ON TABLE {tabla_fact} IS "
          "'Una fila por item vendido, sin pedidos cancelados ni no disponibles. Importes en BRL'")
print(f"{tabla_fact}: {spark.table(tabla_fact).count():,} filas")

### 3.1 · Cuadre de Gold contra Silver
`fact_ventas` debe tener exactamente los ítems de los pedidos válidos y los mismos ingresos. Si un join duplicó filas, aquí se detecta.

In [0]:
items_validos = s("items_pedido").join(pedidos.select("pedido_id"), "pedido_id")
esperado = items_validos.agg(F.count("*").alias("filas"),
                             F.sum(F.col("precio") + F.col("flete")).alias("ingresos")).first()
real = spark.table(tabla_fact).agg(F.count("*").alias("filas"),
                                   F.sum("total_item").alias("ingresos")).first()

excluidos = s("pedidos").filter(F.col("estado_pedido").isin(ESTADOS_NO_VENTA)).count()
print(f"Pedidos excluidos (cancelados / no disponibles): {excluidos:,}")
print(f"Silver -> filas {esperado.filas:,} | ingresos {esperado.ingresos:,.2f}")
print(f"Gold   -> filas {real.filas:,} | ingresos {real.ingresos:,.2f}")

if real.filas != esperado.filas or real.ingresos != esperado.ingresos:
    raise Exception("fact_ventas no cuadra con Silver: revisar los joins (posible duplicación)")
print("Cuadre OK")

## 4. Pregunta 1 · ¿Cómo evolucionan las ventas mes a mes?
> Los meses de los extremos del dataset tienen pocos pedidos: están incompletos y no deben compararse como meses completos.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold}.ventas_mensuales
COMMENT 'Una fila por mes de compra: pedidos, clientes únicos, ingresos y ticket promedio en BRL'
AS
SELECT anio_mes,
       count(DISTINCT pedido_id)        AS pedidos,
       count(DISTINCT cliente_unico_id) AS clientes,
       sum(total_item)                  AS ingresos,
       round(try_divide(sum(total_item), count(DISTINCT pedido_id)), 2) AS ticket_promedio
FROM {gold}.fact_ventas
GROUP BY anio_mes
""")

display(spark.table(f"{gold}.ventas_mensuales").orderBy("anio_mes"))

mejor = spark.table(f"{gold}.ventas_mensuales").orderBy(F.desc("ingresos")).first()
print(f"Mejor mes por ingresos: {mejor.anio_mes} con R$ {mejor.ingresos:,.2f} en {mejor.pedidos:,} pedidos")

## 5. Pregunta 2 · ¿Qué categorías generan más ingresos?

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold}.ranking_categorias
COMMENT 'Una fila por categoría: pedidos, ingresos, participación, precio medio y puntaje medio'
AS
SELECT categoria,
       count(DISTINCT pedido_id)     AS pedidos,
       count(*)                      AS items,
       sum(total_item)               AS ingresos,
       round(100 * sum(total_item) / sum(sum(total_item)) OVER (), 2) AS participacion_pct,
       round(avg(precio), 2)         AS precio_promedio,
       round(avg(puntaje_resena), 2) AS puntaje_medio,
       rank() OVER (ORDER BY sum(total_item) DESC) AS puesto
FROM {gold}.fact_ventas
GROUP BY categoria
""")

display(spark.table(f"{gold}.ranking_categorias").orderBy("puesto").limit(10))

## 6. Pregunta 3 · ¿Dónde se entrega más lento?
Se calcula **a nivel pedido** (no ítem), solo con pedidos entregados.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold}.entregas_por_estado
COMMENT 'Una fila por estado del cliente: tiempo medio de entrega, puntualidad y puntaje medio'
AS
WITH pedidos AS (
    SELECT DISTINCT pedido_id, cliente_estado, dias_entrega, entregado_a_tiempo, puntaje_resena
    FROM {gold}.fact_ventas
    WHERE fecha_entrega IS NOT NULL
)
SELECT cliente_estado,
       count(*)                                         AS pedidos_entregados,
       round(avg(dias_entrega), 1)                      AS dias_entrega_promedio,
       round(100 * avg(CAST(entregado_a_tiempo AS INT)), 1) AS pct_a_tiempo,
       round(avg(puntaje_resena), 2)                    AS puntaje_medio
FROM pedidos
GROUP BY cliente_estado
""")

display(spark.table(f"{gold}.entregas_por_estado").orderBy(F.desc("dias_entrega_promedio")))

## 7. Pregunta 4 · ¿Cuánto cae la satisfacción con el retraso?
Pedidos entregados y con reseña, agrupados por tramo de retraso respecto a la fecha prometida.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold}.satisfaccion_por_retraso
COMMENT 'Puntaje medio y porcentaje de reseñas negativas (1 o 2) por tramo de retraso de entrega'
AS
WITH pedidos AS (
    SELECT DISTINCT pedido_id, dias_retraso, puntaje_resena
    FROM {gold}.fact_ventas
    WHERE fecha_entrega IS NOT NULL AND puntaje_resena IS NOT NULL
)
SELECT CASE WHEN dias_retraso <= 0  THEN '1. A tiempo o antes'
            WHEN dias_retraso <= 3  THEN '2. 1 a 3 días tarde'
            WHEN dias_retraso <= 7  THEN '3. 4 a 7 días tarde'
            WHEN dias_retraso <= 14 THEN '4. 8 a 14 días tarde'
            ELSE '5. Más de 14 días tarde' END              AS tramo_retraso,
       count(*)                                             AS pedidos,
       round(avg(puntaje_resena), 2)                        AS puntaje_medio,
       round(100 * avg(CASE WHEN puntaje_resena <= 2 THEN 1 ELSE 0 END), 1) AS pct_resenas_negativas
FROM pedidos
GROUP BY ALL
""")

display(spark.table(f"{gold}.satisfaccion_por_retraso").orderBy("tramo_retraso"))

## 8. Gobierno de Gold
- **Vista pública**: expone las ventas sin `cliente_unico_id` ni ciudad del cliente. Se dan permisos sobre la vista, no sobre la tabla, y no se duplican datos.
- **Mínimo privilegio**: se concede lectura de las tablas agregadas y de la vista, **no** de `fact_ventas`. En una empresa el permiso iría a un grupo concreto (por ejemplo `analistas_ecommerce`) en lugar de `account users`.

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {gold}.v_ventas_publico
COMMENT 'Ventas sin datos personales del cliente. Apta para consumo general'
AS
SELECT pedido_id, item_num, estado_pedido, fecha_compra, anio_mes, cliente_estado,
       categoria, vendedor_estado, precio, flete, total_item,
       dias_entrega, dias_retraso, entregado_a_tiempo, puntaje_resena
FROM {gold}.fact_ventas
""")

TABLAS_GOLD = ["fact_ventas", "ventas_mensuales", "ranking_categorias",
               "entregas_por_estado", "satisfaccion_por_retraso"]
for t in TABLAS_GOLD:
    spark.sql(f"ALTER TABLE {gold}.{t} SET TAGS ('capa' = 'gold', 'dominio' = 'ecommerce')")
spark.sql(f"ALTER TABLE {gold}.fact_ventas ALTER COLUMN cliente_unico_id SET TAGS ('sensibilidad' = 'personal')")

PUBLICO = ["ventas_mensuales", "ranking_categorias", "entregas_por_estado",
           "satisfaccion_por_retraso", "v_ventas_publico"]
try:
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {gold} TO `account users`")
    for t in PUBLICO:
        spark.sql(f"GRANT SELECT ON TABLE {gold}.{t} TO `account users`")
    print("Lectura concedida a account users sobre:", ", ".join(PUBLICO))
except Exception as e:
    print("No se pudieron conceder permisos (revisar privilegios del usuario):", str(e)[:200])

In [0]:
display(spark.sql(f"SHOW GRANTS ON SCHEMA {gold}"))

display(spark.sql(f"""
    SELECT table_schema AS capa, table_name AS tabla, table_type AS tipo, comment AS descripcion
    FROM {catalogo}.information_schema.tables
    WHERE table_schema IN ('bronze', 'silver', 'gold')
    ORDER BY capa, tabla"""))

## 9. Respuestas de negocio
*Resultados de la ejecución del 28/11/2026 sobre 112.101 ítems vendidos (R$ 15,7 millones); se excluyen 1.234 pedidos cancelados o no disponibles.*

1. **Evolución mensual**: el mejor mes es **noviembre de 2017**, con **R$ 1.172.192** en **7.421 pedidos** (el pico coincide con Black Friday). La tendencia es de **crecimiento sostenido durante 2017**: de R$ 137 mil en enero a más de R$ 760 mil en octubre. Desde enero de 2018 se **estabiliza entre R$ 1,0 y 1,16 millones al mes**. El ticket promedio se mantiene entre R$ 146 y R$ 177. Los meses 2016-09, 2016-12 y 2018-09 tienen 1 o 2 pedidos: están incompletos y no se comparan.

2. **Categorías**: el top 3 (**health_beauty 9,1 %, watches_gifts 8,3 % y bed_bath_table 7,9 %**) concentra el **25,3 %** de los ingresos. La mejor calificada del top 10 es **cool_stuff (4,16)**, seguida de health_beauty (4,15). La peor calificada es **bed_bath_table (3,90)**, pese a ser la categoría con más pedidos (9.399).

3. **Entregas**: los estados más lentos están en el norte del país: **Roraima, RR (29,3 días, 87,8 % a tiempo), Amapá, AP (27,2 días, 97,0 %) y Amazonas, AM (26,4 días, 97,2 %)**, frente a **8,7 días en São Paulo**, que concentra 40.494 pedidos entregados. El estado menos puntual es **Alagoas, AL: 24,5 días y solo 78,6 % a tiempo**, seguido de Maranhão, MA (82,6 %).

4. **Satisfacción**: a tiempo = **4,29** de puntaje (9,2 % de reseñas negativas). Con 1 a 3 días de retraso cae a 3,29, y con 4 a 7 días a 2,11. Con más de 14 días de retraso = **1,73** (**78,2 % de reseñas negativas**). El 93,3 % de los pedidos llega a tiempo, pero el 6,7 % que llega tarde concentra la mayor parte de las malas calificaciones.

**Recomendación:** la satisfacción depende sobre todo de **cumplir la fecha prometida**, más que de la rapidez: estados lentos pero puntuales como AP y AM tienen buen puntaje (4,24). Se recomienda:
1. **Ajustar la fecha estimada** en los estados con baja puntualidad (AL, MA, SE), prometiendo plazos realistas.
2. Crear una **alerta temprana** para pedidos que vayan a pasar de 3 días de retraso (avisar al cliente o compensarlo), porque desde ahí el puntaje cae por debajo de 3.
3. **Preparar stock y logística para noviembre** (Black Friday), el mes de mayor demanda.
4. Revisar la calidad y los tiempos de los vendedores de **bed_bath_table**: es la categoría con más pedidos y la peor calificada del top 10.